In [1]:
import sys, json, random
from pathlib import Path
from collections import defaultdict
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src").exists():   # walk up until we find the repo root
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.prompts import build_messages, format_answer
from src.evaluate import score_prediction
from src.abstention import build_abstention_records

SPLITS = ROOT / "notebooks/data/splits"
OUT = ROOT / "notebooks/data/dpo"
OUT.mkdir(parents=True, exist_ok=True)

pool = [json.loads(l) for l in open(SPLITS / "dpo_pool.jsonl", encoding="utf-8")]
pool_by_id = {r["id"]: r for r in pool}
random.seed(3407)
print(len(pool), "records")
print(pool[0]["query"], "\n->", format_answer(pool[0]["answers"]))

5863 records
Is 'https://www.apple.com' available in the Wayback Machine on September 9, 2015? 
-> [{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]


In [2]:
from openai import OpenAI   # pip install openai if missing
client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
MODEL = "sft-toolcaller"
print(client.models.list().data[0].id)   # check vLLM is up

def sample(record, n=4, temperature=0.8):
    resp = client.chat.completions.create(
        model=MODEL, messages=build_messages(record),
        n=n, temperature=temperature, top_p=0.95, max_tokens=512,
    )
    return [c.message.content for c in resp.choices]

rec = pool[0]
for t in sample(rec):
    print(t)

sft-toolcaller
[{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]
[{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]
[{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]
[{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]


In [3]:
for t in sample(rec):
    s = score_prediction(t, rec)
    print(s["exact_match"], s["failure_type"], "|", t[:120])

1 correct | [{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]
1 correct | [{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]
1 correct | [{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]
1 correct | [{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]


In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def sample_many(records, out_path, n=4, workers=16):
    out_path = Path(out_path)
    done = set()
    if out_path.exists():
        done = {json.loads(l)["id"] for l in open(out_path, encoding="utf-8")}
    todo = [r for r in records if r["id"] not in done]
    print(f"{len(done)} done, {len(todo)} to go")

    def work(r):
        try:
            return {"id": r["id"], "responses": sample(r, n=n)}
        except Exception as e:            # e.g. prompt longer than max-model-len
            return {"id": r["id"], "responses": [], "error": str(e)[:200]}

    with open(out_path, "a", encoding="utf-8") as f, ThreadPoolExecutor(workers) as ex:
        futures = [ex.submit(work, r) for r in todo]
        for fut in tqdm(as_completed(futures), total=len(futures)):
            f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n")
            f.flush()

# smoke test first, then the full run (it skips what's already done)
sample_many(pool[:50], OUT / "samples_pool.jsonl")
# sample_many(pool, OUT / "samples_pool.jsonl")

50 done, 0 to go


0it [00:00, ?it/s]

In [5]:
samples = [json.loads(l) for l in open(OUT / "samples_pool.jsonl", encoding="utf-8")]
rows = []
for s in samples:
    rec = pool_by_id[s["id"]]
    for t in s["responses"]:
        sc = score_prediction(t, rec)
        rows.append({"id": s["id"], "text": t.strip(), "exact": sc["exact_match"], "failure_type": sc["failure_type"]})
df = pd.DataFrame(rows)

print("sample accuracy:", round(df.exact.mean() * 100, 1), "%")
print(df[df.exact == 0].failure_type.value_counts())

def make_pair(rec, chosen, rejected, kind):
    return {"id": rec["id"], "kind": kind, "prompt": build_messages(rec),
            "chosen":   [{"role": "assistant", "content": chosen}],
            "rejected": [{"role": "assistant", "content": rejected}]}

mistake_pairs = []
for rid, g in df.groupby("id"):
    wrong, right = g[g.exact == 0], g[g.exact == 1]
    if wrong.empty:
        continue                                   # model already nails it, nothing to teach
    rec = pool_by_id[rid]
    chosen = right.text.iloc[0] if len(right) else format_answer(rec["answers"])
    bad = wrong.sample(1, random_state=0).iloc[0]
    p = make_pair(rec, chosen, bad.text, "mistake")
    p["failure_type"] = bad.failure_type
    mistake_pairs.append(p)

# balance: cap each failure type so wrong_value doesn't drown the rest
CAP = 500
by_type = defaultdict(list)
for p in mistake_pairs:
    by_type[p["failure_type"]].append(p)
mistake_pairs = []
for ft, ps in by_type.items():
    random.shuffle(ps)
    mistake_pairs += ps[:CAP]
print(len(mistake_pairs), "mistake pairs", {k: min(len(v), CAP) for k, v in by_type.items()})

sample accuracy: 85.5 %
failure_type
wrong_value         25
wrong_call_count     4
Name: count, dtype: int64
11 mistake pairs {'wrong_value': 10, 'wrong_call_count': 1}


In [6]:
abstain_recs = build_abstention_records(pool, n=1000, seed=3407)
abstain_by_id = {r["id"]: r for r in abstain_recs}
sample_many(abstain_recs, OUT / "samples_abstain.jsonl")

abstain_pairs = []
for l in open(OUT / "samples_abstain.jsonl", encoding="utf-8"):
    s = json.loads(l)
    rec = abstain_by_id[s["id"]]
    invented = [t.strip() for t in s["responses"]
                if score_prediction(t, rec)["failure_type"] == "called_tool_when_none_needed"]
    if invented:
        p = make_pair(rec, "[]", invented[0], "abstain")
        p["source_id"] = rec["source_id"]
        abstain_pairs.append(p)
print(len(abstain_pairs), "abstain pairs")

0 done, 1000 to go


  0%|          | 0/1000 [00:00<?, ?it/s]

999 abstain pairs


In [7]:
balance_pairs = []
for p in abstain_pairs:
    rec = pool_by_id[p["source_id"]]               # the ORIGINAL record: same query, real tools
    balance_pairs.append(make_pair(rec, format_answer(rec["answers"]), "[]", "balance"))
print(len(balance_pairs), "balance pairs")

999 balance pairs


In [8]:
# keep abstain at ~18% of the total
n_abs = int(0.18 * len(mistake_pairs) / (1 - 2 * 0.18))
abstain_pairs, balance_pairs = abstain_pairs[:n_abs], balance_pairs[:n_abs]

all_pairs = mistake_pairs + abstain_pairs + balance_pairs
group = lambda p: p.get("source_id", p["id"])      # keep a query's pairs together
groups = sorted({group(p) for p in all_pairs})
random.shuffle(groups)
val_groups = set(groups[: int(0.05 * len(groups))])

train = [p for p in all_pairs if group(p) not in val_groups]
val   = [p for p in all_pairs if group(p) in val_groups]
random.shuffle(train)

for name, rows in [("dpo_train", train), ("dpo_val", val)]:
    with open(OUT / f"{name}.jsonl", "w", encoding="utf-8") as f:
        for p in rows:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(pd.Series([p["kind"] for p in all_pairs]).value_counts())
print(len(train), "train /", len(val), "val")
ex = random.choice(train)
print(ex["kind"], "\nCHOSEN:  ", ex["chosen"][0]["content"], "\nREJECTED:", ex["rejected"][0]["content"])

mistake    11
abstain     3
balance     3
Name: count, dtype: int64
17 train / 0 val
abstain 
CHOSEN:   [] 
REJECTED: [{"name": "get_all_food", "arguments": {"page": "?page=2"}}]
